# PV Register Harmonisation

This notebook maps the nine Austrian photovoltaic register files to official municipalities using only:

- the nine state-specific Anlagenregister workbooks, and
- the Statistik Austria `ortsliste.xls`.

The workflow:

1. loads and normalises the official locality reference,
2. loads all photovoltaic register rows,
3. performs deterministic municipality matching,
4. groups the remaining unresolved place-name variants,
5. creates an editable Excel review workbook,
6. preserves manual review decisions across notebook runs,
7. applies only municipality assignments explicitly marked as `approved`,
8. aggregates unit counts and installed capacity to all Austrian municipalities,
9. validates and exports the harmonised results.

Original register fields are retained unchanged.


## 1. Install packages

In [1]:
%pip install pandas openpyxl xlrd pyarrow

  Using cached xlrd-2.0.2-py2.py3-none-any.whl.metadata (3.5 kB)
Using cached xlrd-2.0.2-py2.py3-none-any.whl (96 kB)
Note: you may need to restart the kernel to use updated packages.


## 2. Imports and paths

In [2]:
from pathlib import Path
from difflib import SequenceMatcher
import re
import unicodedata

import numpy as np
import pandas as pd
from IPython.display import display

from openpyxl.formatting.rule import FormulaRule
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.worksheet.table import Table, TableStyleInfo
from openpyxl.utils import get_column_letter


CURRENT_DIR = Path.cwd()

if (CURRENT_DIR / "data").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "data").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    raise FileNotFoundError(
        "Could not find the project data folder. "
        "Run the notebook from the project root or the notebooks folder."
    )

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

ORTSLISTE_PATH = DATA_DIR / "ortsliste.xls"

OUTPUT_XLSX_PATH = OUTPUT_DIR / "pv_municipality_harmonised.xlsx"
PLANT_OUTPUT_PATH = OUTPUT_DIR / "pv_plants_harmonised.parquet"

# This workbook is created by the notebook and then edited manually.
# Existing decisions are read before the workbook is regenerated.
PLACE_REVIEW_PATH = OUTPUT_DIR / "place_name_alias_review.xlsx"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT.resolve())
print("Data folder:", DATA_DIR.resolve())
print("Output folder:", OUTPUT_DIR.resolve())
print("Review workbook:", PLACE_REVIEW_PATH.resolve())

Project root: C:\Users\friha\TU Wien\Diplomarbeit\code\at-lv-scenario-benchmark\harmonisation\harmonisation_PV
Data folder: C:\Users\friha\TU Wien\Diplomarbeit\code\at-lv-scenario-benchmark\harmonisation\harmonisation_PV\data
Output folder: C:\Users\friha\TU Wien\Diplomarbeit\code\at-lv-scenario-benchmark\harmonisation\harmonisation_PV\outputs
Review workbook: C:\Users\friha\TU Wien\Diplomarbeit\code\at-lv-scenario-benchmark\harmonisation\harmonisation_PV\outputs\place_name_alias_review.xlsx


## 3. Configuration

In [3]:
REFERENCE_DATE = pd.Timestamp("2026-07-18")

REGISTER_FILES = {
    "B": DATA_DIR / "Anlagenregister_Burgenland.xlsx",
    "K": DATA_DIR / "Anlagenregister_Kärnten.xlsx",
    "NO": DATA_DIR / "Anlagenregister_Niederösterreich.xlsx",
    "OO": DATA_DIR / "Anlagenregister_Oberösterreich.xlsx",
    "S": DATA_DIR / "Anlagenregister_Salzburg.xlsx",
    "ST": DATA_DIR / "Anlagenregister_Steiermark.xlsx",
    "T": DATA_DIR / "Anlagenregister_Tirol.xlsx",
    "V": DATA_DIR / "Anlagenregister_Vorarlberg.xlsx",
    "W": DATA_DIR / "Anlagenregister_Wien.xlsx",
}

GKZ_PREFIX_TO_STATE_CODE = {
    "1": "B",
    "2": "K",
    "3": "NO",
    "4": "OO",
    "5": "S",
    "6": "ST",
    "7": "T",
    "8": "V",
    "9": "W",
}

VIENNA_MUNICIPALITY_ID = "90001"
VIENNA_MUNICIPALITY_NAME = "Wien"
VIENNA_POSTCODES = {
    str(value)
    for value in range(1010, 1240, 10)
}

PV_TECHNOLOGY_VALUES = {
    "photovoltaik",
}

# LV scope proxy:
# ElWG § 97(1) assigns generation units with a grid-effective power
# ("netzwirksame Leistung") of up to and including 100 kW to grid level 7.
# The E-Control register provides bottleneck capacity ("Engpassleistung")
# instead of grid-effective power. Therefore, bottleneck capacity is used
# as a conservative proxy for the LV scope.
LV_PROXY_MAX_BOTTLENECK_CAPACITY_KW = 100.0

# These thresholds do not exclude rows from the review workbook.
# They only assign a review priority.
REVIEW_PRIORITY_UNIT_COUNT = 10
REVIEW_PRIORITY_CAPACITY_KW = 100.0
REVIEW_MAX_CANDIDATES = 3

REVIEW_STATUS_VALUES = [
    "pending_review",
    "pending_name_postcode_conflict",
    "pending_no_candidate",
    "approved",
    "rejected",
    "ambiguous",
    "insufficient_information",
]

assert ORTSLISTE_PATH.exists(), (
    f"Ortsliste not found: {ORTSLISTE_PATH.resolve()}"
)

missing_registers = [
    path.name
    for path in REGISTER_FILES.values()
    if not path.exists()
]

assert not missing_registers, (
    "Missing register files: "
    + ", ".join(missing_registers)
)

print("All required input files were found.")

All required input files were found.


## 4. General helper functions

In [4]:
def clean_id(value, width):
    """Return a numeric identifier as a zero-padded string."""
    if pd.isna(value):
        return pd.NA

    text = str(value).strip()

    if re.fullmatch(r"\d+(?:\.0+)?", text):
        text = text.split(".")[0]
    else:
        text = re.sub(r"\D", "", text)

    if not text:
        return pd.NA

    return text.zfill(width)


def extract_postcodes(value):
    """Extract all standalone four-digit postcodes from a value."""
    if pd.isna(value):
        return []

    return re.findall(
        r"(?<!\d)\d{4}(?!\d)",
        str(value),
    )


def normalize_postcode(value):
    """Return the first valid four-digit postcode from a value."""
    postcodes = extract_postcodes(value)
    return postcodes[0] if postcodes else pd.NA


def normalize_name(value):
    """Normalise an Austrian place name for deterministic matching.

    Only general, place-independent transformations are performed.
    Municipality-specific assignments are handled through the manually
    reviewed mapping workbook.
    """
    if pd.isna(value):
        return pd.NA

    text = unicodedata.normalize(
        "NFKC",
        str(value),
    ).casefold()

    # General character normalisation.
    text = (
        text
        .replace("ä", "ae")
        .replace("ö", "oe")
        .replace("ü", "ue")
        .replace("ß", "ss")
    )

    # Remove a postcode included at the beginning of the place field.
    # Example: "9020 Klagenfurt" -> "Klagenfurt"
    text = re.sub(
        r"^\s*\d{4}\s*[,;:-]?\s*",
        "",
        text,
    )

    # Remove a trailing country reference.
    # Example: "Villach, Österreich" -> "Villach"
    text = re.sub(
        r"\s*,?\s*oesterreich\s*$",
        "",
        text,
    )

    # Expand general Austrian place-name abbreviations.
    text = re.sub(
        r"\bst\b\.?\s*",
        "sankt ",
        text,
    )
    text = re.sub(
        r"\ba\b\.?\s*d\b\.?\s*",
        "an der ",
        text,
    )
    text = re.sub(
        r"\ba\b\.?\s*m\b\.?\s*",
        "am ",
        text,
    )

    # Expand only stable regional abbreviations.
    text = re.sub(
        r"\bi\b\.?\s*b\b\.?",
        "im burgenland ",
        text,
    )
    text = re.sub(
        r"\bi\b\.?\s*lavanttal\b",
        "im lavanttal ",
        text,
    )

    # Standardise separators and punctuation.
    text = re.sub(
        r"[-_/—]",
        " ",
        text,
    )
    text = re.sub(
        r"[^\w\s]",
        " ",
        text,
        flags=re.UNICODE,
    )
    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

    return text if text else pd.NA


def parse_capacity_kw(value):
    """Convert a raw capacity value to kilowatts."""
    if pd.isna(value):
        return np.nan

    if isinstance(value, (int, float, np.number)):
        return float(value)

    text = str(value).strip()
    text = (
        text
        .replace("\u00a0", "")
        .replace(" ", "")
    )

    if "," in text and "." in text:
        if text.rfind(",") > text.rfind("."):
            text = (
                text
                .replace(".", "")
                .replace(",", ".")
            )
        else:
            text = text.replace(",", "")
    elif "," in text:
        text = text.replace(",", ".")

    text = re.sub(
        r"[^0-9.\-]",
        "",
        text,
    )

    try:
        return float(text)
    except ValueError:
        return np.nan


def normalize_register_state(value):
    """Map a federal-state label to the internal state code."""
    if pd.isna(value):
        return pd.NA

    text = unicodedata.normalize(
        "NFKD",
        str(value),
    )
    text = "".join(
        character
        for character in text
        if not unicodedata.combining(character)
    )
    text = (
        text
        .upper()
        .strip()
        .replace(".", "")
        .replace(" ", "")
    )

    mapping = {
        "B": "B",
        "BURGENLAND": "B",
        "K": "K",
        "KAERNTEN": "K",
        "KARNTEN": "K",
        "NO": "NO",
        "NOE": "NO",
        "NIEDEROESTERREICH": "NO",
        "NIEDEROSTERREICH": "NO",
        "OO": "OO",
        "OOE": "OO",
        "OBEROESTERREICH": "OO",
        "OBEROSTERREICH": "OO",
        "S": "S",
        "SALZBURG": "S",
        "ST": "ST",
        "STEIERMARK": "ST",
        "T": "T",
        "TIROL": "T",
        "V": "V",
        "VORARLBERG": "V",
        "W": "W",
        "WIEN": "W",
    }

    return mapping.get(text, pd.NA)


def aggregate_mapping_method(series):
    """Summarise plant-level mapping methods for one municipality."""
    methods = sorted(
        set(series.dropna())
    )

    if len(methods) == 0:
        return "not_applicable"

    if len(methods) == 1:
        return methods[0]

    return "mixed"


def build_unique_lookup(frame, key_columns):
    """Build a lookup containing only unambiguous municipality matches."""
    required_columns = (
        list(key_columns)
        + ["municipality_id"]
    )

    missing_columns = [
        column
        for column in required_columns
        if column not in frame.columns
    ]

    if missing_columns:
        raise KeyError(
            "Missing lookup columns: "
            + ", ".join(missing_columns)
        )

    clean = frame.dropna(
        subset=required_columns,
    ).copy()

    grouped = (
        clean
        .groupby(
            key_columns,
            dropna=False,
        )["municipality_id"]
        .agg(
            lambda values: tuple(
                pd.unique(values)
            )
        )
    )

    return {
        (
            key
            if isinstance(key, tuple)
            else (key,)
        ): municipality_ids[0]
        for key, municipality_ids in grouped.items()
        if len(municipality_ids) == 1
    }


def apply_lookup(
    frame,
    lookup,
    key_columns,
    method,
):
    """Apply a municipality lookup to currently unresolved rows."""
    unresolved_index = frame.index[
        frame["municipality_id"].isna()
    ]

    if len(unresolved_index) == 0:
        return

    keys = [
        tuple(values)
        for values in frame.loc[
            unresolved_index,
            key_columns,
        ].itertuples(
            index=False,
            name=None,
        )
    ]

    results = pd.Series(
        [
            lookup.get(
                key,
                pd.NA,
            )
            for key in keys
        ],
        index=unresolved_index,
        dtype="string",
    )

    matched_index = results.index[
        results.notna()
    ]

    if len(matched_index) == 0:
        return

    frame.loc[
        matched_index,
        "municipality_id",
    ] = results.loc[
        matched_index
    ]

    frame.loc[
        matched_index,
        "derivation_method",
    ] = method


def lookup_results(
    frame,
    lookup,
    key_columns,
):
    """Return municipality lookup results without modifying the frame."""
    keys = [
        tuple(values)
        for values in frame[
            key_columns
        ].itertuples(
            index=False,
            name=None,
        )
    ]

    return pd.Series(
        [
            lookup.get(
                key,
                pd.NA,
            )
            for key in keys
        ],
        index=frame.index,
        dtype="string",
    )


def name_similarity(left, right):
    """Return a deterministic string-similarity score from zero to one."""
    if pd.isna(left) or pd.isna(right):
        return np.nan

    return SequenceMatcher(
        None,
        str(left),
        str(right),
    ).ratio()


def join_unique_examples(values, maximum=5):
    """Join a limited number of distinct raw source values."""
    examples = [
        str(value).strip()
        for value in pd.unique(values.dropna())
        if str(value).strip()
    ]

    return " | ".join(
        examples[:maximum]
    )

## 5. Review-workbook helper functions

In [5]:
def assign_review_priority(
    unit_count,
    installed_capacity_kw,
):
    """Classify the importance of one unresolved group for review."""
    high_units = (
        pd.notna(unit_count)
        and unit_count >= REVIEW_PRIORITY_UNIT_COUNT
    )
    high_capacity = (
        pd.notna(installed_capacity_kw)
        and installed_capacity_kw
        >= REVIEW_PRIORITY_CAPACITY_KW
    )

    if high_units and high_capacity:
        return "high_units_and_capacity"

    if high_units:
        return "high_units"

    if high_capacity:
        return "high_capacity"

    return "standard"


def build_review_candidates(
    unresolved_groups,
    locality_reference,
    max_candidates=3,
):
    """Create ranked official candidates for every unresolved group.

    Candidate generation uses only the official Ortsliste and the grouped
    unresolved register values. It does not approve any assignment.
    """
    official_candidates = (
        locality_reference[
            [
                "federal_state_code",
                "postcode",
                "municipality_id",
                "municipality_name",
                "municipality_name_normalized",
                "locality_name",
                "locality_name_normalized",
            ]
        ]
        .drop_duplicates()
        .copy()
    )

    state_postcode_pools = {
        key: group.reset_index(drop=True)
        for key, group in official_candidates.groupby(
            [
                "federal_state_code",
                "postcode",
            ],
            dropna=False,
        )
    }

    national_postcode_pools = {
        key: group.reset_index(drop=True)
        for key, group in official_candidates.groupby(
            "postcode",
            dropna=False,
        )
    }

    state_pools = {
        key: group.reset_index(drop=True)
        for key, group in official_candidates.groupby(
            "federal_state_code",
            dropna=False,
        )
    }

    output_rows = []

    for _, unresolved_group in unresolved_groups.iterrows():
        state_code = unresolved_group[
            "source_federal_state_code"
        ]
        postcode = unresolved_group["postcode"]
        place_name_normalized = unresolved_group[
            "place_name_normalized"
        ]

        candidates = state_postcode_pools.get(
            (state_code, postcode)
        )
        candidate_scope = "state_and_postcode"

        if candidates is None or candidates.empty:
            candidates = national_postcode_pools.get(
                postcode
            )
            candidate_scope = "national_postcode"

        if candidates is None or candidates.empty:
            candidates = state_pools.get(
                state_code
            )
            candidate_scope = "state_name_search"

        row = {
            **unresolved_group.to_dict(),
            "candidate_scope": candidate_scope,
            "review_status": (
                "pending_name_postcode_conflict"
                if (
                    pd.notna(
                        unresolved_group.get(
                            "mapping_issue"
                        )
                    )
                    and unresolved_group.get(
                        "mapping_issue"
                    ) == "name_postcode_conflict"
                )
                else "pending_review"
            ),
            "approved_municipality_id": pd.NA,
            "review_note": pd.NA,
        }

        if candidates is None or candidates.empty:
            row["review_status"] = "pending_no_candidate"

            for rank in range(
                1,
                max_candidates + 1,
            ):
                row[
                    f"suggested_municipality_id_{rank}"
                ] = pd.NA
                row[
                    f"suggested_municipality_name_{rank}"
                ] = pd.NA
                row[
                    f"suggested_locality_name_{rank}"
                ] = pd.NA
                row[
                    f"suggested_similarity_score_{rank}"
                ] = np.nan

            output_rows.append(row)
            continue

        ranked_candidates = candidates.copy()

        ranked_candidates[
            "municipality_score"
        ] = ranked_candidates[
            "municipality_name_normalized"
        ].map(
            lambda value: name_similarity(
                place_name_normalized,
                value,
            )
        )

        ranked_candidates[
            "locality_score"
        ] = ranked_candidates[
            "locality_name_normalized"
        ].map(
            lambda value: name_similarity(
                place_name_normalized,
                value,
            )
        )

        ranked_candidates[
            "candidate_score"
        ] = ranked_candidates[
            [
                "municipality_score",
                "locality_score",
            ]
        ].max(axis=1)

        ranked_candidates = (
            ranked_candidates
            .sort_values(
                [
                    "candidate_score",
                    "municipality_id",
                ],
                ascending=[
                    False,
                    True,
                ],
            )
            .drop_duplicates(
                "municipality_id"
            )
            .head(max_candidates)
            .reset_index(drop=True)
        )

        for rank in range(
            1,
            max_candidates + 1,
        ):
            candidate_index = rank - 1

            if candidate_index < len(
                ranked_candidates
            ):
                candidate = ranked_candidates.iloc[
                    candidate_index
                ]

                row[
                    f"suggested_municipality_id_{rank}"
                ] = candidate[
                    "municipality_id"
                ]
                row[
                    f"suggested_municipality_name_{rank}"
                ] = candidate[
                    "municipality_name"
                ]
                row[
                    f"suggested_locality_name_{rank}"
                ] = candidate[
                    "locality_name"
                ]
                row[
                    f"suggested_similarity_score_{rank}"
                ] = candidate[
                    "candidate_score"
                ]
            else:
                row[
                    f"suggested_municipality_id_{rank}"
                ] = pd.NA
                row[
                    f"suggested_municipality_name_{rank}"
                ] = pd.NA
                row[
                    f"suggested_locality_name_{rank}"
                ] = pd.NA
                row[
                    f"suggested_similarity_score_{rank}"
                ] = np.nan

        output_rows.append(row)

    review_table = pd.DataFrame(
        output_rows
    )

    priority_order = {
        "high_units_and_capacity": 0,
        "high_units": 1,
        "high_capacity": 2,
        "standard": 3,
    }

    review_table["_priority_order"] = (
        review_table[
            "review_priority"
        ].map(priority_order)
    )

    review_table = (
        review_table
        .sort_values(
            [
                "_priority_order",
                "unit_count",
                "installed_capacity_kw",
            ],
            ascending=[
                True,
                False,
                False,
            ],
        )
        .drop(
            columns="_priority_order"
        )
        .reset_index(drop=True)
    )

    return review_table


def read_review_table(review_path):
    """Read the manually editable Excel review workbook."""
    if not review_path.exists():
        return pd.DataFrame()

    try:
        return pd.read_excel(
            review_path,
            sheet_name="review",
            dtype="string",
            engine="openpyxl",
        )
    except ValueError as error:
        raise ValueError(
            "The review workbook does not contain "
            "a worksheet named 'review'."
        ) from error


def merge_existing_review_decisions(
    generated_candidates,
    review_path,
):
    """Preserve manual decisions from an existing review workbook."""
    review_key = [
        "source_federal_state_code",
        "postcode",
        "place_name_normalized",
    ]

    if not review_path.exists():
        return generated_candidates

    existing = read_review_table(
        review_path
    )

    if existing.empty:
        return generated_candidates

    decision_columns = [
        *review_key,
        "review_status",
        "approved_municipality_id",
        "review_note",
    ]

    missing_columns = [
        column
        for column in decision_columns
        if column not in existing.columns
    ]

    if missing_columns:
        raise ValueError(
            "Existing review workbook is missing columns: "
            + ", ".join(missing_columns)
        )

    decisions = (
        existing[
            decision_columns
        ]
        .drop_duplicates(
            subset=review_key,
            keep="last",
        )
    )

    merged = generated_candidates.merge(
        decisions,
        on=review_key,
        how="left",
        suffixes=(
            "",
            "_existing",
        ),
        validate="one_to_one",
    )

    for column in [
        "review_status",
        "approved_municipality_id",
        "review_note",
    ]:
        existing_column = (
            f"{column}_existing"
        )

        merged[column] = (
            merged[existing_column]
            .combine_first(
                merged[column]
            )
        )

        merged = merged.drop(
            columns=existing_column
        )

    return merged


def write_review_workbook(
    review_table,
    review_path,
):
    """Write a filterable and manually editable review workbook."""
    review_export = review_table.copy()

    instructions = pd.DataFrame(
        {
            "Field or step": [
                "Purpose",
                "First run",
                "review_status",
                "approved_municipality_id",
                "review_note",
                "Approval rule",
                "Do not approve",
                "Rerun",
                "Important",
            ],
            "Instruction": [
                (
                    "The workbook contains automatically generated "
                    "municipality candidates for unresolved register "
                    "place-name groups."
                ),
                (
                    "No candidate is approved automatically. Review the "
                    "rows in the 'review' sheet."
                ),
                (
                    "Set this field to 'approved' only after confirming "
                    "the municipality against the official Ortsliste."
                ),
                (
                    "Enter the official five-digit municipality ID (GKZ)."
                ),
                (
                    "Document the basis of the decision, for example: "
                    "'Short municipality name; state and postcode checked'."
                ),
                (
                    "Use the source state, postcode, place-name relation "
                    "and official Ortsliste together."
                ),
                (
                    "Do not approve contradictory, ambiguous or "
                    "insufficiently documented rows."
                ),
                (
                    "Save and close this workbook, then rerun the notebook. "
                    "Existing decisions are preserved."
                ),
                (
                    "Do not change source fields or automatically generated "
                    "candidate columns."
                ),
            ],
        }
    )

    try:
        with pd.ExcelWriter(
            review_path,
            engine="openpyxl",
        ) as writer:
            review_export.to_excel(
                writer,
                sheet_name="review",
                index=False,
            )

            instructions.to_excel(
                writer,
                sheet_name="instructions",
                index=False,
            )

            workbook = writer.book
            review_sheet = writer.sheets[
                "review"
            ]
            instruction_sheet = writer.sheets[
                "instructions"
            ]

            review_sheet.freeze_panes = "A2"
            review_sheet.auto_filter.ref = (
                review_sheet.dimensions
            )
            review_sheet.sheet_view.showGridLines = False

            instruction_sheet.freeze_panes = "A2"
            instruction_sheet.sheet_view.showGridLines = False

            header_fill = PatternFill(
                fill_type="solid",
                fgColor="1F4E78",
            )
            header_font = Font(
                bold=True,
                color="FFFFFF",
            )

            for worksheet in [
                review_sheet,
                instruction_sheet,
            ]:
                for cell in worksheet[1]:
                    cell.fill = header_fill
                    cell.font = header_font
                    cell.alignment = Alignment(
                        horizontal="center",
                        vertical="center",
                        wrap_text=True,
                    )

            editable_fill = PatternFill(
                fill_type="solid",
                fgColor="FFF2CC",
            )

            column_positions = {
                cell.value: cell.column
                for cell in review_sheet[1]
            }

            editable_columns = [
                "review_status",
                "approved_municipality_id",
                "review_note",
            ]

            for column_name in editable_columns:
                column_number = (
                    column_positions.get(
                        column_name
                    )
                )

                if column_number is None:
                    continue

                for row_number in range(
                    2,
                    review_sheet.max_row + 1,
                ):
                    review_sheet.cell(
                        row=row_number,
                        column=column_number,
                    ).fill = editable_fill

            text_columns = [
                "source_federal_state_code",
                "postcode",
                "postcode_raw_examples",
                "approved_municipality_id",
                "suggested_municipality_id_1",
                "suggested_municipality_id_2",
                "suggested_municipality_id_3",
            ]

            for column_name in text_columns:
                column_number = (
                    column_positions.get(
                        column_name
                    )
                )

                if column_number is None:
                    continue

                for row_number in range(
                    2,
                    review_sheet.max_row + 1,
                ):
                    review_sheet.cell(
                        row=row_number,
                        column=column_number,
                    ).number_format = "@"

            numeric_formats = {
                "unit_count": "#,##0",
                "installed_capacity_kw": "#,##0.000",
                "suggested_similarity_score_1": "0.0000",
                "suggested_similarity_score_2": "0.0000",
                "suggested_similarity_score_3": "0.0000",
            }

            for (
                column_name,
                number_format,
            ) in numeric_formats.items():
                column_number = (
                    column_positions.get(
                        column_name
                    )
                )

                if column_number is None:
                    continue

                for row_number in range(
                    2,
                    review_sheet.max_row + 1,
                ):
                    review_sheet.cell(
                        row=row_number,
                        column=column_number,
                    ).number_format = number_format

            review_status_column = (
                column_positions.get(
                    "review_status"
                )
            )

            if review_status_column is not None:
                status_validation = DataValidation(
                    type="list",
                    formula1=(
                        '"'
                        + ",".join(
                            REVIEW_STATUS_VALUES
                        )
                        + '"'
                    ),
                    allow_blank=False,
                )

                status_validation.error = (
                    "Select one of the configured "
                    "review-status values."
                )
                status_validation.errorTitle = (
                    "Invalid review status"
                )
                status_validation.prompt = (
                    "Set to approved only after "
                    "manual verification."
                )
                status_validation.promptTitle = (
                    "Review status"
                )

                review_sheet.add_data_validation(
                    status_validation
                )

                status_column_letter = (
                    get_column_letter(
                        review_status_column
                    )
                )

                status_validation.add(
                    f"{status_column_letter}2:"
                    f"{status_column_letter}"
                    f"{review_sheet.max_row}"
                )

                approved_rule = FormulaRule(
                    formula=[
                        (
                            f'${status_column_letter}2'
                            '="approved"'
                        )
                    ],
                    fill=PatternFill(
                        fill_type="solid",
                        fgColor="D9EAD3",
                    ),
                )

                rejected_rule = FormulaRule(
                    formula=[
                        (
                            f'${status_column_letter}2'
                            '="rejected"'
                        )
                    ],
                    fill=PatternFill(
                        fill_type="solid",
                        fgColor="F4CCCC",
                    ),
                )

                ambiguous_rule = FormulaRule(
                    formula=[
                        (
                            f'OR('
                            f'${status_column_letter}2="ambiguous",'
                            f'${status_column_letter}2='
                            '"insufficient_information")'
                        )
                    ],
                    fill=PatternFill(
                        fill_type="solid",
                        fgColor="FCE5CD",
                    ),
                )

                review_sheet.conditional_formatting.add(
                    f"A2:"
                    f"{get_column_letter(review_sheet.max_column)}"
                    f"{review_sheet.max_row}",
                    approved_rule,
                )
                review_sheet.conditional_formatting.add(
                    f"A2:"
                    f"{get_column_letter(review_sheet.max_column)}"
                    f"{review_sheet.max_row}",
                    rejected_rule,
                )
                review_sheet.conditional_formatting.add(
                    f"A2:"
                    f"{get_column_letter(review_sheet.max_column)}"
                    f"{review_sheet.max_row}",
                    ambiguous_rule,
                )

            if review_sheet.max_row >= 2:
                review_table_object = Table(
                    displayName="PlaceNameReviewTable",
                    ref=review_sheet.dimensions,
                )
                review_table_object.tableStyleInfo = TableStyleInfo(
                    name="TableStyleMedium2",
                    showFirstColumn=False,
                    showLastColumn=False,
                    showRowStripes=True,
                    showColumnStripes=False,
                )
                review_sheet.add_table(
                    review_table_object
                )

            width_by_name = {
                "source_federal_state_code": 12,
                "postcode": 10,
                "place_name_normalized": 30,
                "place_name_raw_examples": 38,
                "postcode_raw_examples": 20,
                "mapping_issue": 28,
                "review_priority": 24,
                "unit_count": 12,
                "installed_capacity_kw": 18,
                "candidate_scope": 22,
                "review_status": 34,
                "approved_municipality_id": 24,
                "review_note": 52,
            }

            for column_number in range(
                1,
                review_sheet.max_column + 1,
            ):
                column_name = review_sheet.cell(
                    row=1,
                    column=column_number,
                ).value

                default_width = 24

                if (
                    isinstance(column_name, str)
                    and "suggested_municipality_name"
                    in column_name
                ):
                    default_width = 30
                elif (
                    isinstance(column_name, str)
                    and "suggested_locality_name"
                    in column_name
                ):
                    default_width = 30
                elif (
                    isinstance(column_name, str)
                    and "suggested_similarity_score"
                    in column_name
                ):
                    default_width = 16
                elif (
                    isinstance(column_name, str)
                    and "suggested_municipality_id"
                    in column_name
                ):
                    default_width = 18

                review_sheet.column_dimensions[
                    get_column_letter(
                        column_number
                    )
                ].width = width_by_name.get(
                    column_name,
                    default_width,
                )

            for row in review_sheet.iter_rows():
                for cell in row:
                    cell.alignment = Alignment(
                        vertical="top",
                        wrap_text=True,
                    )

            instruction_sheet.column_dimensions[
                "A"
            ].width = 30
            instruction_sheet.column_dimensions[
                "B"
            ].width = 95

            for row in instruction_sheet.iter_rows():
                for cell in row:
                    cell.alignment = Alignment(
                        vertical="top",
                        wrap_text=True,
                    )

    except PermissionError as error:
        raise PermissionError(
            "The review workbook could not be written. "
            "Close place_name_alias_review.xlsx in Excel "
            "and rerun the cell."
        ) from error


def load_approved_aliases(
    review_path,
    municipality_master,
):
    """Load and validate manually approved municipality mappings."""
    output_columns = [
        "source_federal_state_code",
        "postcode",
        "place_name_normalized",
        "municipality_id",
        "review_note",
    ]

    if not review_path.exists():
        return pd.DataFrame(
            columns=output_columns
        )

    review = read_review_table(
        review_path
    )

    required_columns = {
        "source_federal_state_code",
        "postcode",
        "place_name_normalized",
        "review_status",
        "approved_municipality_id",
        "review_note",
    }

    missing_columns = (
        required_columns
        - set(review.columns)
    )

    if missing_columns:
        raise ValueError(
            "Review workbook is missing columns: "
            + ", ".join(
                sorted(missing_columns)
            )
        )

    approved = review[
        review["review_status"].eq(
            "approved"
        )
    ].copy()

    if approved.empty:
        return pd.DataFrame(
            columns=output_columns
        )

    missing_approved_id = approved[
        "approved_municipality_id"
    ].isna()

    if missing_approved_id.any():
        raise ValueError(
            "At least one row is marked as approved "
            "without an approved municipality ID."
        )

    missing_review_note = (
        approved["review_note"]
        .fillna("")
        .str.strip()
        .eq("")
    )

    if missing_review_note.any():
        raise ValueError(
            "Every approved row requires a review note."
        )

    approved[
        "source_federal_state_code"
    ] = approved[
        "source_federal_state_code"
    ].map(normalize_register_state)

    approved["postcode"] = approved[
        "postcode"
    ].map(normalize_postcode)

    approved[
        "place_name_normalized"
    ] = approved[
        "place_name_normalized"
    ].map(normalize_name)

    approved["municipality_id"] = approved[
        "approved_municipality_id"
    ].map(
        lambda value: clean_id(
            value,
            5,
        )
    )

    official_ids = set(
        municipality_master[
            "municipality_id"
        ]
    )

    unknown_ids = (
        set(
            approved[
                "municipality_id"
            ].dropna()
        )
        - official_ids
    )

    if unknown_ids:
        raise ValueError(
            "Unknown municipality IDs in review workbook: "
            + ", ".join(
                sorted(unknown_ids)
            )
        )

    key_columns = [
        "source_federal_state_code",
        "postcode",
        "place_name_normalized",
    ]

    conflicts = (
        approved
        .groupby(
            key_columns,
            dropna=False,
        )["municipality_id"]
        .nunique()
    )

    if (conflicts > 1).any():
        raise ValueError(
            "Conflicting approved municipality assignments "
            "exist for the same review key."
        )

    return (
        approved[
            key_columns
            + [
                "municipality_id",
                "review_note",
            ]
        ]
        .drop_duplicates()
        .reset_index(drop=True)
    )

## 6. Load the Statistik Austria Ortsliste

In [6]:
raw_preview = pd.read_excel(
    ORTSLISTE_PATH,
    header=None,
    engine="xlrd",
)

header_row = None

for row_index in range(
    min(15, len(raw_preview))
):
    values = [
        str(value).strip().casefold()
        for value in raw_preview.iloc[
            row_index
        ].tolist()
    ]

    checks = [
        any(
            "gemeinde" in value
            and "kennziffer" in value
            for value in values
        ),
        any(
            "gemeindename" in value
            for value in values
        ),
        any(
            "ortschaft" in value
            and "kennziffer" in value
            for value in values
        ),
        any(
            "ortschaftsname" in value
            for value in values
        ),
        any(
            "postleitzahl" in value
            for value in values
        ),
    ]

    if sum(checks) >= 4:
        header_row = row_index
        break

assert header_row is not None, (
    "The Ortsliste header row was not found."
)

ort_raw = pd.read_excel(
    ORTSLISTE_PATH,
    header=header_row,
    engine="xlrd",
)

ort_raw.columns = [
    str(column).strip()
    for column in ort_raw.columns
]

column_municipality_id = next(
    column
    for column in ort_raw.columns
    if "gemeinde" in column.casefold()
    and "kennziffer" in column.casefold()
)

column_municipality_name = next(
    column
    for column in ort_raw.columns
    if column.casefold() == "gemeindename"
)

column_locality_id = next(
    column
    for column in ort_raw.columns
    if "ortschaft" in column.casefold()
    and "kennziffer" in column.casefold()
)

column_locality_name = next(
    column
    for column in ort_raw.columns
    if "ortschaftsname" in column.casefold()
)

column_postcode = next(
    column
    for column in ort_raw.columns
    if "postleitzahl" in column.casefold()
)

ort = pd.DataFrame(
    {
        "municipality_id": (
            ort_raw[
                column_municipality_id
            ].map(
                lambda value: clean_id(
                    value,
                    5,
                )
            )
        ),
        "municipality_name": (
            ort_raw[
                column_municipality_name
            ]
            .astype("string")
            .str.strip()
        ),
        "locality_id": (
            ort_raw[
                column_locality_id
            ].map(
                lambda value: clean_id(
                    value,
                    8,
                )
            )
        ),
        "locality_name": (
            ort_raw[
                column_locality_name
            ]
            .astype("string")
            .str.strip()
        ),
        "postcode_list": (
            ort_raw[
                column_postcode
            ].map(extract_postcodes)
        ),
    }
)

ort = ort[
    ort["municipality_id"].str.fullmatch(
        r"\d{5}",
        na=False,
    )
].copy()

ort["federal_state_code"] = (
    ort["municipality_id"]
    .str[0]
    .map(GKZ_PREFIX_TO_STATE_CODE)
)

ort[
    "municipality_name_normalized"
] = ort[
    "municipality_name"
].map(normalize_name)

ort[
    "locality_name_normalized"
] = ort[
    "locality_name"
].map(normalize_name)

locality_reference = (
    ort.explode("postcode_list")
    .rename(
        columns={
            "postcode_list": "postcode"
        }
    )
    .reset_index(drop=True)
)

locality_reference["postcode"] = (
    locality_reference["postcode"]
    .astype("string")
    .str.strip()
)

locality_reference = locality_reference[
    locality_reference[
        "postcode"
    ].str.fullmatch(
        r"\d{4}",
        na=False,
    )
].copy()

print(
    "Locality rows:",
    len(locality_reference),
)
print(
    "Municipalities in Ortsliste:",
    locality_reference[
        "municipality_id"
    ].nunique(),
)

Locality rows: 18877
Municipalities in Ortsliste: 2091


## 7. Create the municipality master table

In [7]:
municipality_master = (
    locality_reference[
        [
            "municipality_id",
            "municipality_name",
            "federal_state_code",
        ]
    ]
    .drop_duplicates()
    .sort_values("municipality_id")
    .reset_index(drop=True)
)

vienna_master_row = pd.DataFrame(
    {
        "municipality_id": [
            VIENNA_MUNICIPALITY_ID
        ],
        "municipality_name": [
            VIENNA_MUNICIPALITY_NAME
        ],
        "federal_state_code": [
            "W"
        ],
    }
)

municipality_master = (
    pd.concat(
        [
            municipality_master,
            vienna_master_row,
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        "municipality_id"
    )
    .sort_values(
        "municipality_id"
    )
    .reset_index(drop=True)
)

assert municipality_master[
    "municipality_id"
].is_unique

assert len(municipality_master) == 2092

assert (
    municipality_master[
        "municipality_id"
    ]
    == VIENNA_MUNICIPALITY_ID
).sum() == 1

display(
    municipality_master.head()
)

,municipality_id,municipality_name,federal_state_code
0,10101,Eisenstadt,B
1,10201,Rust,B
2,10301,Breitenbrunn am Neusiedler See,B
3,10302,Donnerskirchen,B
4,10303,Großhöflein,B


## 8. Build official municipality lookups

In [8]:
locality_pair_lookup = build_unique_lookup(
    locality_reference,
    [
        "federal_state_code",
        "postcode",
        "locality_name_normalized",
    ],
)

municipality_pair_lookup = build_unique_lookup(
    locality_reference,
    [
        "federal_state_code",
        "postcode",
        "municipality_name_normalized",
    ],
)

national_locality_pair_lookup = build_unique_lookup(
    locality_reference,
    [
        "postcode",
        "locality_name_normalized",
    ],
)

national_municipality_pair_lookup = build_unique_lookup(
    locality_reference,
    [
        "postcode",
        "municipality_name_normalized",
    ],
)

unique_locality_lookup = build_unique_lookup(
    locality_reference,
    [
        "federal_state_code",
        "locality_name_normalized",
    ],
)

unique_municipality_lookup = build_unique_lookup(
    locality_reference,
    [
        "federal_state_code",
        "municipality_name_normalized",
    ],
)

unique_postcode_lookup = build_unique_lookup(
    locality_reference,
    [
        "federal_state_code",
        "postcode",
    ],
)

municipality_postcode_sets = (
    locality_reference
    .groupby(
        "municipality_id"
    )["postcode"]
    .agg(
        lambda values: set(
            values.dropna()
        )
    )
    .to_dict()
)

municipality_name_lookup = (
    municipality_master
    .set_index(
        "municipality_id"
    )["municipality_name"]
    .to_dict()
)

municipality_state_lookup = (
    municipality_master
    .set_index(
        "municipality_id"
    )["federal_state_code"]
    .to_dict()
)

print(
    "Unique state locality + postcode keys:",
    len(locality_pair_lookup),
)
print(
    "Unique state municipality + postcode keys:",
    len(municipality_pair_lookup),
)
print(
    "Unique national locality + postcode keys:",
    len(national_locality_pair_lookup),
)
print(
    "Unique national municipality + postcode keys:",
    len(national_municipality_pair_lookup),
)
print(
    "Unique locality names within state:",
    len(unique_locality_lookup),
)
print(
    "Unique municipality names within state:",
    len(unique_municipality_lookup),
)
print(
    "Unique postcodes within state:",
    len(unique_postcode_lookup),
)

Unique state locality + postcode keys: 18455
Unique state municipality + postcode keys: 3938
Unique national locality + postcode keys: 18451
Unique national municipality + postcode keys: 3938
Unique locality names within state: 12333
Unique municipality names within state: 2091
Unique postcodes within state: 1269


## 9. Load all photovoltaic register files

In [9]:
CAPACITY_COLUMN = (
    "Engpassleistung (kW <sub>el</sub>)"
)

register_frames = []
register_report_rows = []
technology_summary_frames = []
excluded_solar_like_frames = []

# Audit tables for rows excluded from the LV-oriented dataset.
excluded_above_lv_capacity_frames = []
invalid_capacity_frames = []


for (
    expected_state_code,
    path,
) in REGISTER_FILES.items():

    # ---------------------------------------------------------------------
    # Load source workbook.
    # ---------------------------------------------------------------------

    register_raw = pd.read_excel(
        path,
        sheet_name=0,
        engine="openpyxl",
    )

    required_columns = [
        "ID",
        "Plz",
        "Ort",
        "Bundesland",
        "Technologie",
        CAPACITY_COLUMN,
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in register_raw.columns
    ]

    assert not missing_columns, (
        f"Missing columns in {path.name}: "
        f"{missing_columns}"
    )


    # ---------------------------------------------------------------------
    # Preserve source-row information for auditing.
    # ---------------------------------------------------------------------

    register_raw[
        "source_row_number"
    ] = np.arange(
        2,
        len(register_raw) + 2,
    )


    # ---------------------------------------------------------------------
    # Normalise technology field.
    # ---------------------------------------------------------------------

    register_raw[
        "technology_normalized"
    ] = register_raw[
        "Technologie"
    ].map(normalize_name)


    # ---------------------------------------------------------------------
    # Store complete source technology distribution.
    # ---------------------------------------------------------------------

    technology_summary = (
        register_raw
        .groupby(
            [
                "Technologie",
                "technology_normalized",
            ],
            dropna=False,
        )
        .size()
        .reset_index(
            name="row_count"
        )
    )

    technology_summary[
        "source_file"
    ] = path.name

    technology_summary_frames.append(
        technology_summary
    )


    # ---------------------------------------------------------------------
    # Select photovoltaic rows.
    # ---------------------------------------------------------------------

    pv_mask = register_raw[
        "technology_normalized"
    ].isin(
        PV_TECHNOLOGY_VALUES
    )


    # ---------------------------------------------------------------------
    # Identify solar-like names outside the explicit PV whitelist.
    # ---------------------------------------------------------------------

    solar_like_mask = (
        register_raw[
            "technology_normalized"
        ]
        .astype("string")
        .str.contains(
            "solar|photovolta",
            case=False,
            na=False,
        )
    )

    excluded_solar_like = register_raw[
        solar_like_mask
        & ~pv_mask
    ].copy()

    if not excluded_solar_like.empty:

        excluded_solar_like[
            "source_file"
        ] = path.name

        excluded_solar_like[
            "expected_state_code"
        ] = expected_state_code

        excluded_solar_like_frames.append(
            excluded_solar_like
        )


    # ---------------------------------------------------------------------
    # Continue only with accepted PV technology rows.
    # ---------------------------------------------------------------------

    register = register_raw[
        pv_mask
    ].copy()


    # ---------------------------------------------------------------------
    # Normalise and validate federal-state information.
    # ---------------------------------------------------------------------

    register[
        "source_federal_state_code"
    ] = register[
        "Bundesland"
    ].map(
        normalize_register_state
    )

    found_state_codes = set(
        register[
            "source_federal_state_code"
        ]
        .dropna()
        .unique()
    )

    assert found_state_codes == {
        expected_state_code
    }, (
        f"Unexpected state code in "
        f"{path.name}: "
        f"{found_state_codes}"
    )


    # ---------------------------------------------------------------------
    # Preserve source identifiers and original values.
    # ---------------------------------------------------------------------

    register[
        "source_file"
    ] = path.name

    register[
        "source_register_id"
    ] = register[
        "ID"
    ].astype("string")

    register[
        "source_row_key"
    ] = (
        expected_state_code
        + "::"
        + register[
            "source_row_number"
        ].astype("string")
    )

    register[
        "postcode_raw"
    ] = register[
        "Plz"
    ]

    register[
        "place_name_raw"
    ] = register[
        "Ort"
    ]

    register[
        "capacity_raw"
    ] = register[
        CAPACITY_COLUMN
    ]


    # ---------------------------------------------------------------------
    # Normalise spatial information.
    # ---------------------------------------------------------------------

    register[
        "postcode"
    ] = register[
        "Plz"
    ].map(
        normalize_postcode
    )

    register[
        "place_name"
    ] = (
        register[
            "Ort"
        ]
        .astype("string")
        .str.strip()
    )

    register[
        "place_name_normalized"
    ] = register[
        "Ort"
    ].map(
        normalize_name
    )


    # ---------------------------------------------------------------------
    # Parse bottleneck capacity.
    # ---------------------------------------------------------------------

    register[
        "installed_capacity_kw"
    ] = register[
        CAPACITY_COLUMN
    ].map(
        parse_capacity_kw
    )

    register[
        "reference_date"
    ] = REFERENCE_DATE


    # =====================================================================
    # LV SCOPE CLASSIFICATION
    # =====================================================================
    #
    # Legal basis:
    #
    # ElWG § 97(1):
    #   installations with a grid-effective power
    #   ("netzwirksame Leistung") of up to and including 100 kW are
    #   generally assigned to grid level 7.
    #
    # ElWG § 106:
    #   grid level 7 corresponds to low voltage (<= 1 kV).
    #
    # Data limitation:
    #
    # The E-Control plant register provides bottleneck capacity
    # ("Engpassleistung"), not grid-effective power.
    #
    # Therefore, bottleneck capacity <= 100 kW is used as a conservative
    # proxy for identifying PV installations relevant to the LV scope.
    #
    # The filter does NOT claim to reconstruct the actual connection level
    # of every individual installation.
    # =====================================================================

    capacity = register[
        "installed_capacity_kw"
    ]


    # Category 1: missing, zero or negative capacity.
    invalid_capacity_mask = (
        capacity.isna()
        | capacity.le(0)
    )


    # Category 2: positive capacity above the LV proxy threshold.
    excluded_above_lv_capacity_mask = (
        capacity.gt(
            LV_PROXY_MAX_BOTTLENECK_CAPACITY_KW
        )
    )


    # Category 3: valid capacity within the LV proxy scope.
    included_lv_proxy_mask = (
        capacity.gt(0)
        & capacity.le(
            LV_PROXY_MAX_BOTTLENECK_CAPACITY_KW
        )
    )


    # Every accepted PV row must belong to exactly one category.
    scope_category_count = (
        invalid_capacity_mask.astype(int)
        + excluded_above_lv_capacity_mask.astype(int)
        + included_lv_proxy_mask.astype(int)
    )

    assert scope_category_count.eq(1).all(), (
        f"LV scope classification failed for {path.name}: "
        "at least one PV row belongs to zero or multiple categories."
    )


    # =====================================================================
    # AUDIT: INVALID CAPACITY
    # =====================================================================

    invalid_capacity = register[
        invalid_capacity_mask
    ].copy()

    invalid_capacity[
        "lv_scope_status"
    ] = "invalid_capacity"

    invalid_capacity[
        "lv_scope_reason"
    ] = (
        "missing_or_non_positive_bottleneck_capacity"
    )

    if not invalid_capacity.empty:
        invalid_capacity_frames.append(
            invalid_capacity
        )


    # =====================================================================
    # AUDIT: CAPACITY ABOVE 100 kW
    # =====================================================================

    excluded_above_lv_capacity = register[
        excluded_above_lv_capacity_mask
    ].copy()

    excluded_above_lv_capacity[
        "lv_scope_status"
    ] = "excluded_above_100kw"

    excluded_above_lv_capacity[
        "lv_scope_reason"
    ] = (
        "bottleneck_capacity_above_lv_proxy_threshold"
    )

    if not excluded_above_lv_capacity.empty:
        excluded_above_lv_capacity_frames.append(
            excluded_above_lv_capacity
        )


    # =====================================================================
    # DATA RETAINED FOR LV HARMONISATION
    # =====================================================================

    register_in_scope = register[
        included_lv_proxy_mask
    ].copy()

    register_in_scope[
        "lv_scope_status"
    ] = "included_lv_proxy"

    register_in_scope[
        "lv_scope_reason"
    ] = (
        "positive_bottleneck_capacity_at_or_below_100kw"
    )


    # =====================================================================
    # SOURCE STATISTICS
    # =====================================================================

    positive_capacity_mask = (
        capacity.gt(0)
    )

    positive_capacity_before_filter = (
        capacity.loc[
            positive_capacity_mask
        ].sum(
            min_count=1
        )
    )

    lv_proxy_capacity = (
        capacity.loc[
            included_lv_proxy_mask
        ].sum(
            min_count=1
        )
    )

    excluded_above_100kw_capacity = (
        capacity.loc[
            excluded_above_lv_capacity_mask
        ].sum(
            min_count=1
        )
    )


    register_report_rows.append(
        {
            "source_file": (
                path.name
            ),

            "source_federal_state_code": (
                expected_state_code
            ),

            "raw_rows": (
                len(register_raw)
            ),

            # Kept for backwards compatibility with earlier reports.
            "pv_rows": (
                len(register)
            ),

            "pv_rows_before_lv_scope_filter": (
                len(register)
            ),

            "lv_proxy_rows": int(
                included_lv_proxy_mask.sum()
            ),

            "excluded_above_100kw_rows": int(
                excluded_above_lv_capacity_mask.sum()
            ),

            "invalid_capacity_rows": int(
                invalid_capacity_mask.sum()
            ),

            "positive_capacity_kw_before_lv_scope_filter": (
                positive_capacity_before_filter
            ),

            "lv_proxy_capacity_kw": (
                lv_proxy_capacity
            ),

            "excluded_above_100kw_capacity_kw": (
                excluded_above_100kw_capacity
            ),

            "excluded_solar_like_rows": (
                len(excluded_solar_like)
            ),

            # Spatial-data quality for rows entering municipality mapping.
            "missing_postcode": int(
                register_in_scope[
                    "postcode"
                ].isna().sum()
            ),

            "missing_place_name": int(
                register_in_scope[
                    "place_name_normalized"
                ].isna().sum()
            ),

            # Capacity quality before the LV filter.
            "missing_capacity": int(
                capacity.isna().sum()
            ),

            "non_positive_capacity": int(
                capacity.le(0).sum()
            ),
        }
    )


    # Only rows inside the LV proxy scope proceed to municipality mapping.
    register_frames.append(
        register_in_scope
    )


# =========================================================================
# COMBINE ALL NINE FEDERAL STATES
# =========================================================================

pv_units = pd.concat(
    register_frames,
    ignore_index=True,
    sort=False,
)

register_report = pd.DataFrame(
    register_report_rows
)

technology_summary = pd.concat(
    technology_summary_frames,
    ignore_index=True,
)


# =========================================================================
# COMBINE AUDIT TABLES
# =========================================================================

if excluded_solar_like_frames:
    excluded_solar_like_rows = pd.concat(
        excluded_solar_like_frames,
        ignore_index=True,
        sort=False,
    )
else:
    excluded_solar_like_rows = pd.DataFrame()


if excluded_above_lv_capacity_frames:
    excluded_above_lv_capacity_rows = pd.concat(
        excluded_above_lv_capacity_frames,
        ignore_index=True,
        sort=False,
    )
else:
    excluded_above_lv_capacity_rows = pd.DataFrame()


if invalid_capacity_frames:
    invalid_capacity_rows = pd.concat(
        invalid_capacity_frames,
        ignore_index=True,
        sort=False,
    )
else:
    invalid_capacity_rows = pd.DataFrame()


# =========================================================================
# CALCULATE LV SCOPE SHARES BY FEDERAL STATE
# =========================================================================

register_report[
    "lv_proxy_unit_share"
] = (
    register_report[
        "lv_proxy_rows"
    ]
    / register_report[
        "pv_rows_before_lv_scope_filter"
    ]
)


register_report[
    "lv_proxy_capacity_share"
] = (
    register_report[
        "lv_proxy_capacity_kw"
    ]
    / register_report[
        "positive_capacity_kw_before_lv_scope_filter"
    ]
)


# =========================================================================
# AUSTRIA-WIDE LV SCOPE SUMMARY
# =========================================================================

lv_scope_summary = pd.DataFrame(
    [
        {
            "scope": "Austria",

            "pv_rows_before_lv_scope_filter": int(
                register_report[
                    "pv_rows_before_lv_scope_filter"
                ].sum()
            ),

            "lv_proxy_rows": int(
                register_report[
                    "lv_proxy_rows"
                ].sum()
            ),

            "excluded_above_100kw_rows": int(
                register_report[
                    "excluded_above_100kw_rows"
                ].sum()
            ),

            "invalid_capacity_rows": int(
                register_report[
                    "invalid_capacity_rows"
                ].sum()
            ),

            "positive_capacity_kw_before_lv_scope_filter": (
                register_report[
                    "positive_capacity_kw_before_lv_scope_filter"
                ].sum()
            ),

            "lv_proxy_capacity_kw": (
                register_report[
                    "lv_proxy_capacity_kw"
                ].sum()
            ),

            "excluded_above_100kw_capacity_kw": (
                register_report[
                    "excluded_above_100kw_capacity_kw"
                ].sum()
            ),
        }
    ]
)


lv_scope_summary[
    "lv_proxy_unit_share"
] = (
    lv_scope_summary[
        "lv_proxy_rows"
    ]
    / lv_scope_summary[
        "pv_rows_before_lv_scope_filter"
    ]
)


lv_scope_summary[
    "lv_proxy_capacity_share"
] = (
    lv_scope_summary[
        "lv_proxy_capacity_kw"
    ]
    / lv_scope_summary[
        "positive_capacity_kw_before_lv_scope_filter"
    ]
)


# =========================================================================
# BASIC CONSISTENCY CHECKS
# =========================================================================

pv_rows_before_filter = int(
    register_report[
        "pv_rows_before_lv_scope_filter"
    ].sum()
)

classified_pv_rows = int(
    register_report[
        "lv_proxy_rows"
    ].sum()
    + register_report[
        "excluded_above_100kw_rows"
    ].sum()
    + register_report[
        "invalid_capacity_rows"
    ].sum()
)

assert (
    pv_rows_before_filter
    == classified_pv_rows
), (
    "PV row counts before and after LV scope classification do not match."
)


assert pv_units[
    "installed_capacity_kw"
].gt(0).all(), (
    "The LV proxy dataset contains non-positive capacities."
)


assert pv_units[
    "installed_capacity_kw"
].le(
    LV_PROXY_MAX_BOTTLENECK_CAPACITY_KW
).all(), (
    "The LV proxy dataset contains PV installations above 100 kW."
)


# =========================================================================
# OUTPUT SUMMARY
# =========================================================================

print(
    "PV rows before LV scope filter:",
    pv_rows_before_filter,
)

print(
    "PV rows retained in LV proxy scope:",
    len(pv_units),
)

print(
    "PV rows excluded above 100 kW:",
    int(
        register_report[
            "excluded_above_100kw_rows"
        ].sum()
    ),
)

print(
    "PV rows with invalid capacity:",
    int(
        register_report[
            "invalid_capacity_rows"
        ].sum()
    ),
)

print(
    "LV proxy unit share:",
    f"{lv_scope_summary.loc[0, 'lv_proxy_unit_share']:.2%}",
)

print(
    "LV proxy capacity share:",
    f"{lv_scope_summary.loc[0, 'lv_proxy_capacity_share']:.2%}",
)

print(
    "Excluded capacity above 100 kW [MW]:",
    f"{lv_scope_summary.loc[0, 'excluded_above_100kw_capacity_kw'] / 1000:.3f}",
)


display(
    register_report
)

display(
    lv_scope_summary
)

PV rows before LV scope filter: 581075
PV rows retained in LV proxy scope: 568997
PV rows excluded above 100 kW: 11769
PV rows with invalid capacity: 309
LV proxy unit share: 97.92%
LV proxy capacity share: 56.90%
Excluded capacity above 100 kW [MW]: 5004.296


,source_file,source_federal_state_code,raw_rows,pv_rows,pv_rows_before_lv_scope_filter,lv_proxy_rows,excluded_above_100kw_rows,invalid_capacity_rows,positive_capacity_kw_before_lv_scope_filter,lv_proxy_capacity_kw,excluded_above_100kw_capacity_kw,excluded_solar_like_rows,missing_postcode,missing_place_name,missing_capacity,non_positive_capacity,lv_proxy_unit_share,lv_proxy_capacity_share
0,Anlagenregister_Burgenland.xlsx,B,33613,33612,33612,33180,431,1,1156582.844,338393.015,818189.829,0,0,0,0,1,0.987147,0.292580
1,Anlagenregister_Kärnten.xlsx,K,42566,42565,42565,41490,1055,20,830417.698,498015.373,332402.325,0,2,0,0,20,0.974745,0.599717
2,Anlagenregister_Niederösterreich.xlsx,NO,168560,168555,168555,165317,3132,106,2839563.772,1733380.203,1106183.569,4,12,12,0,106,0.980790,0.610439
3,Anlagenregister_Oberösterreich.xlsx,OO,121818,121817,121817,119381,2431,5,2672952.765,1595002.200,1077950.565,0,19,17,0,5,0.980003,0.596719
4,Anlagenregister_Salzburg.xlsx,S,34551,34550,34550,33878,665,7,606770.798,414791.167,191979.631,0,0,0,0,7,0.980550,0.683604
5,Anlagenregister_Steiermark.xlsx,ST,96925,96919,96919,94534,2221,164,1989381.187,1031355.419,958025.768,5,2,1,0,164,0.975392,0.518430
6,Anlagenregister_Tirol.xlsx,T,39813,39809,39809,38935,869,5,728575.693,498239.215,230336.478,3,0,0,0,5,0.978045,0.683854
7,Anlagenregister_Vorarlberg.xlsx,V,24364,24363,24363,23933,429,1,410675.072,297799.696,112875.376,0,0,0,0,1,0.982350,0.725147
8,Anlagenregister_Wien.xlsx,W,18886,18885,18885,18349,536,0,376233.938,199881.246,176352.692,0,2,2,0,0,0.971618,0.531269


,scope,pv_rows_before_lv_scope_filter,lv_proxy_rows,excluded_above_100kw_rows,invalid_capacity_rows,positive_capacity_kw_before_lv_scope_filter,lv_proxy_capacity_kw,excluded_above_100kw_capacity_kw,lv_proxy_unit_share,lv_proxy_capacity_share
0,Austria,581075,568997,11769,309,1.161115e+07,6606857.534,5004296.233,0.979214,0.56901


## 10. Automatic municipality mapping

In [10]:
def run_automatic_mapping(pv_units):
    """Run all deterministic non-reviewed municipality rules."""
    mapped = pv_units.copy()

    mapped[
        "municipality_id"
    ] = pd.Series(
        pd.NA,
        index=mapped.index,
        dtype="string",
    )

    mapped[
        "derivation_method"
    ] = "unresolved"

    mapped[
        "mapping_issue"
    ] = pd.NA

    # Vienna is absent from the official Ortsliste.
    # Only records with a Vienna postcode and a compatible source/name
    # indication are assigned to municipality 90001.
    vienna_mask = (
        mapped[
            "postcode"
        ].isin(VIENNA_POSTCODES)
        & (
            mapped[
                "source_federal_state_code"
            ].eq("W")
            | mapped[
                "place_name_normalized"
            ].eq("wien")
        )
    )

    mapped.loc[
        vienna_mask,
        "municipality_id",
    ] = VIENNA_MUNICIPALITY_ID

    mapped.loc[
        vienna_mask,
        "derivation_method",
    ] = "vienna_postcode_and_source_or_name"

    apply_lookup(
        mapped,
        locality_pair_lookup,
        [
            "source_federal_state_code",
            "postcode",
            "place_name_normalized",
        ],
        "exact_locality_name_postcode",
    )

    apply_lookup(
        mapped,
        municipality_pair_lookup,
        [
            "source_federal_state_code",
            "postcode",
            "place_name_normalized",
        ],
        "exact_municipality_name_postcode",
    )

    apply_lookup(
        mapped,
        national_locality_pair_lookup,
        [
            "postcode",
            "place_name_normalized",
        ],
        "exact_national_locality_name_postcode",
    )

    apply_lookup(
        mapped,
        national_municipality_pair_lookup,
        [
            "postcode",
            "place_name_normalized",
        ],
        "exact_national_municipality_name_postcode",
    )

    unresolved_mask = mapped[
        "municipality_id"
    ].isna()

    locality_candidates = lookup_results(
        mapped.loc[
            unresolved_mask
        ],
        unique_locality_lookup,
        [
            "source_federal_state_code",
            "place_name_normalized",
        ],
    )

    municipality_candidates = lookup_results(
        mapped.loc[
            unresolved_mask
        ],
        unique_municipality_lookup,
        [
            "source_federal_state_code",
            "place_name_normalized",
        ],
    )

    mapped.loc[
        locality_candidates.index,
        "_locality_candidate_id",
    ] = locality_candidates

    mapped.loc[
        municipality_candidates.index,
        "_municipality_candidate_id",
    ] = municipality_candidates

    def postcode_is_compatible(
        postcode,
        municipality_id,
    ):
        if pd.isna(
            municipality_id
        ):
            return False

        if pd.isna(postcode):
            return True

        return postcode in (
            municipality_postcode_sets.get(
                municipality_id,
                set(),
            )
        )

    locality_compatible = mapped.apply(
        lambda row: postcode_is_compatible(
            row["postcode"],
            row.get(
                "_locality_candidate_id"
            ),
        ),
        axis=1,
    )

    municipality_compatible = mapped.apply(
        lambda row: postcode_is_compatible(
            row["postcode"],
            row.get(
                "_municipality_candidate_id"
            ),
        ),
        axis=1,
    )

    eligible_locality = (
        mapped[
            "municipality_id"
        ].isna()
        & mapped[
            "_locality_candidate_id"
        ].notna()
        & locality_compatible
    )

    mapped.loc[
        eligible_locality,
        "municipality_id",
    ] = mapped.loc[
        eligible_locality,
        "_locality_candidate_id",
    ]

    mapped.loc[
        eligible_locality,
        "derivation_method",
    ] = (
        "exact_unique_locality_name_"
        "postcode_compatible"
    )

    eligible_municipality = (
        mapped[
            "municipality_id"
        ].isna()
        & mapped[
            "_municipality_candidate_id"
        ].notna()
        & municipality_compatible
    )

    mapped.loc[
        eligible_municipality,
        "municipality_id",
    ] = mapped.loc[
        eligible_municipality,
        "_municipality_candidate_id",
    ]

    mapped.loc[
        eligible_municipality,
        "derivation_method",
    ] = (
        "exact_unique_municipality_name_"
        "postcode_compatible"
    )

    name_conflict_mask = (
        mapped[
            "municipality_id"
        ].isna()
        & mapped[
            "postcode"
        ].notna()
        & (
            (
                mapped[
                    "_locality_candidate_id"
                ].notna()
                & ~locality_compatible
            )
            |
            (
                mapped[
                    "_municipality_candidate_id"
                ].notna()
                & ~municipality_compatible
            )
        )
    )

    mapped.loc[
        name_conflict_mask,
        "mapping_issue",
    ] = "name_postcode_conflict"

    apply_lookup(
        mapped,
        unique_postcode_lookup,
        [
            "source_federal_state_code",
            "postcode",
        ],
        "unique_postcode",
    )

    return mapped.drop(
        columns=[
            "_locality_candidate_id",
            "_municipality_candidate_id",
        ],
        errors="ignore",
    )


mapped_units = run_automatic_mapping(
    pv_units
)

automatic_resolved_mask = mapped_units[
    "municipality_id"
].notna()

print(
    "Automatically mapped PV units:",
    automatic_resolved_mask.sum(),
)
print(
    "Automatically unresolved PV units:",
    (~automatic_resolved_mask).sum(),
)
print(
    "Automatic unit mapping rate:",
    f"{automatic_resolved_mask.mean():.2%}",
)

Automatically mapped PV units: 548002
Automatically unresolved PV units: 20995
Automatic unit mapping rate: 96.31%


## 11. Generate the editable Excel review workbook

This section groups every unresolved register variant and creates municipality candidates using only the Anlagenregister values and the official Ortsliste.

Candidate generation is automatic and **does not constitute an approved assignment**.

The workbook is created at:

`outputs/place_name_alias_review.xlsx`

Only these columns are intended for manual editing:

- `review_status`
- `approved_municipality_id`
- `review_note`

Existing decisions in those columns are preserved when the notebook is rerun. Close the workbook in Excel before running this cell.


In [11]:
unresolved_units_automatic = mapped_units[
    mapped_units[
        "municipality_id"
    ].isna()
].copy()

unresolved_groups = (
    unresolved_units_automatic
    .groupby(
        [
            "source_federal_state_code",
            "postcode",
            "place_name_normalized",
        ],
        dropna=False,
    )
    .agg(
        place_name_raw_examples=(
            "place_name_raw",
            join_unique_examples,
        ),
        postcode_raw_examples=(
            "postcode_raw",
            join_unique_examples,
        ),
        mapping_issue=(
            "mapping_issue",
            lambda values: (
                "name_postcode_conflict"
                if values.eq(
                    "name_postcode_conflict"
                ).any()
                else pd.NA
            ),
        ),
        unit_count=(
            "source_row_key",
            "size",
        ),
        installed_capacity_kw=(
            "installed_capacity_kw",
            lambda values: values.sum(
                min_count=1
            ),
        ),
    )
    .reset_index()
)

unresolved_groups[
    "review_priority"
] = unresolved_groups.apply(
    lambda row: assign_review_priority(
        row["unit_count"],
        row["installed_capacity_kw"],
    ),
    axis=1,
)

generated_review = build_review_candidates(
    unresolved_groups,
    locality_reference,
    max_candidates=(
        REVIEW_MAX_CANDIDATES
    ),
)

review_table = merge_existing_review_decisions(
    generated_review,
    PLACE_REVIEW_PATH,
)

write_review_workbook(
    review_table,
    PLACE_REVIEW_PATH,
)

print(
    "Review groups:",
    len(review_table),
)
print(
    "Review workbook:",
    PLACE_REVIEW_PATH.resolve(),
)

display(
    review_table.head(30)
)

Review groups: 1958
Review workbook: C:\Users\friha\TU Wien\Diplomarbeit\code\at-lv-scenario-benchmark\harmonisation\harmonisation_PV\outputs\place_name_alias_review.xlsx


,source_federal_state_code,postcode,place_name_normalized,place_name_raw_examples,postcode_raw_examples,mapping_issue,unit_count,installed_capacity_kw,review_priority,candidate_scope,...,suggested_locality_name_1,suggested_similarity_score_1,suggested_municipality_id_2,suggested_municipality_name_2,suggested_locality_name_2,suggested_similarity_score_2,suggested_municipality_id_3,suggested_municipality_name_3,suggested_locality_name_3,suggested_similarity_score_3
0,NO,2500,baden bei wien,Baden bei Wien,2500,None,660,5810.990,high_units_and_capacity,state_and_postcode,...,Baden,0.526316,30613,Heiligenkreuz,Siegenfeld,0.250000,30601,Alland,Alland,0.200000
1,ST,8045,graz andritz,Graz-Andritz | Graz Andritz,8045.0,None,572,4976.290,high_units_and_capacity,state_and_postcode,...,"Graz,12.Bez.:Andritz",0.774194,60646,Stattegg,Hochgreit,0.380952,60654,Weinitzen,Oberschöckl,0.380952
2,K,9872,millstatt,Millstatt | 9872 Millstatt,9872.0,None,362,4319.280,high_units_and_capacity,state_and_postcode,...,Obermillstatt,0.818182,20634,Seeboden am Millstätter See,Tangern,0.486486,20630,Radenthein,Starfach,0.352941
3,ST,8063,eggersdorf b graz,Eggersdorf b.Graz | Eggersdorf b. Graz,8063.0,None,356,3115.965,high_units_and_capacity,state_and_postcode,...,Brodersdorf,0.944444,60666,Nestelbach bei Graz,Mitterlaßnitz,0.500000,61727,Ludersdorf-Wilfersdorf,Pircha,0.461538
4,ST,8463,leutschach weinstrasse,Leutschach/Weinstraße,8463.0,None,328,3724.061,high_units_and_capacity,state_and_postcode,...,Eichberg-Trautenburg,0.862745,61013,Großklein,Oberfahrenbach,0.277778,61050,Gamlitz,Steinbach,0.258065
5,K,9020,klagenfurt,Klagenfurt | KLagenfurt | klagenfurt,9020.0,None,323,4769.950,high_units_and_capacity,state_and_postcode,...,"Klagenfurt,01.Bez.:Innere Stadt",0.571429,20442,Magdalensberg,Farchern,0.434783,20418,Maria Saal,Walddorf,0.222222
6,ST,8051,graz goesting,Graz-Gösting | Graz Gösting,8051.0,None,281,2937.920,high_units_and_capacity,state_and_postcode,...,"Graz,13.Bez.:Gösting",0.787879,60648,Thal,Thal,0.117647,<NA>,<NA>,<NA>,NaN
7,ST,8401,kalsdorf b graz,Kalsdorf b.Graz | Kalsdorf b. Graz,8401.0,None,280,2778.175,high_units_and_capacity,state_and_postcode,...,Forst,0.937500,60608,Feldkirchen bei Graz,Abtissendorf,0.514286,60670,Premstätten,Laa,0.222222
8,S,5023,salzburg gnigl,Salzburg-Gnigl | Salzburg Gnigl | Salzburg - G...,5023,None,279,2771.060,high_units_and_capacity,state_and_postcode,...,Salzburg,0.727273,50321,Koppl,Heuberg,0.285714,<NA>,<NA>,<NA>,NaN
9,ST,8511,sankt stefan o stainz,Sankt Stefan o.Stainz | 8511 Sankt Stefan o.St...,8511.0,None,277,2137.330,high_units_and_capacity,state_and_postcode,...,Greisdorf,0.976744,60350,Stainz,Pichling bei Stainz,0.500000,60318,Lannach,Teipl,0.230769


## 12. Manual review and reviewed municipality assignment

Open `outputs/place_name_alias_review.xlsx`.

For an unambiguous row confirmed against the official Ortsliste, enter:

```text
review_status = approved
approved_municipality_id = official five-digit GKZ
review_note = brief documented reason
```

Example:

```text
review_status: approved
approved_municipality_id: 20620
review_note: Short register name; state and postcode checked against Ortsliste.
```

Do not approve a row when the place name and postcode contradict each other, multiple municipalities remain plausible, or the source information is insufficient.

After saving and closing the workbook, rerun the notebook. The next cell applies only approved assignments.


## 13. Apply manually reviewed municipality mappings

In [12]:
approved_aliases = load_approved_aliases(
    PLACE_REVIEW_PATH,
    municipality_master,
)

reviewed_alias_lookup = build_unique_lookup(
    approved_aliases,
    [
        "source_federal_state_code",
        "postcode",
        "place_name_normalized",
    ],
)

# REVIEWED MAPPING:
# Apply only municipality assignments explicitly approved after
# manual comparison with the official Statistik Austria Ortsliste.
apply_lookup(
    mapped_units,
    reviewed_alias_lookup,
    [
        "source_federal_state_code",
        "postcode",
        "place_name_normalized",
    ],
    "reviewed_place_alias",
)

mapped_units[
    "municipality_name"
] = mapped_units[
    "municipality_id"
].map(
    municipality_name_lookup
)

mapped_units[
    "federal_state_code"
] = mapped_units[
    "municipality_id"
].map(
    municipality_state_lookup
)

mapped_units[
    "cross_state_assignment"
] = (
    mapped_units[
        "municipality_id"
    ].notna()
    & mapped_units[
        "federal_state_code"
    ].ne(
        mapped_units[
            "source_federal_state_code"
        ]
    )
)

resolved_mask = mapped_units[
    "municipality_id"
].notna()

print(
    "Approved reviewed mappings:",
    len(approved_aliases),
)
print(
    "Mapped PV units:",
    resolved_mask.sum(),
)
print(
    "Unresolved PV units:",
    (~resolved_mask).sum(),
)
print(
    "Final unit mapping rate:",
    f"{resolved_mask.mean():.2%}",
)
print(
    "Cross-state assignments:",
    mapped_units[
        "cross_state_assignment"
    ].sum(),
)

Approved reviewed mappings: 937
Mapped PV units: 566053
Unresolved PV units: 2944
Final unit mapping rate: 99.48%
Cross-state assignments: 1829


## 14. Review final mapping results

In [13]:
mapping_summary = (
    mapped_units
    .groupby(
        [
            "source_federal_state_code",
            "derivation_method",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        unit_count=(
            "source_row_key",
            "size",
        ),
        installed_capacity_kw=(
            "installed_capacity_kw",
            lambda values: values.sum(
                min_count=1
            ),
        ),
    )
)

state_mapping_rows = []

for (
    state_code,
    group,
) in mapped_units.groupby(
    "source_federal_state_code",
    dropna=False,
):
    mapped = group[
        "municipality_id"
    ].notna()

    total_capacity = group[
        "installed_capacity_kw"
    ].sum(min_count=1)

    mapped_capacity = group.loc[
        mapped,
        "installed_capacity_kw",
    ].sum(min_count=1)

    state_mapping_rows.append(
        {
            "source_federal_state_code": (
                state_code
            ),
            "total_unit_count": len(group),
            "mapped_unit_count": int(
                mapped.sum()
            ),
            "unresolved_unit_count": int(
                (~mapped).sum()
            ),
            "unit_mapping_rate": (
                mapped.mean()
            ),
            "total_capacity_kw": (
                total_capacity
            ),
            "mapped_capacity_kw": (
                mapped_capacity
            ),
            "unresolved_capacity_kw": (
                total_capacity
                - mapped_capacity
            ),
            "capacity_mapping_rate": (
                mapped_capacity
                / total_capacity
                if pd.notna(
                    total_capacity
                )
                and total_capacity != 0
                else np.nan
            ),
            "cross_state_assignment_count": int(
                group[
                    "cross_state_assignment"
                ].sum()
            ),
        }
    )

state_mapping_report = pd.DataFrame(
    state_mapping_rows
)

unresolved_units = mapped_units[
    mapped_units[
        "municipality_id"
    ].isna()
].copy()

unresolved_top_places = (
    unresolved_units
    .groupby(
        [
            "source_federal_state_code",
            "place_name_raw",
            "postcode_raw",
            "mapping_issue",
        ],
        dropna=False,
    )
    .agg(
        unit_count=(
            "source_row_key",
            "size",
        ),
        installed_capacity_kw=(
            "installed_capacity_kw",
            lambda values: values.sum(
                min_count=1
            ),
        ),
    )
    .reset_index()
    .sort_values(
        [
            "source_federal_state_code",
            "unit_count",
            "installed_capacity_kw",
        ],
        ascending=[
            True,
            False,
            False,
        ],
    )
)

display(
    state_mapping_report
)
display(
    mapping_summary
)
display(
    unresolved_top_places.head(30)
)

,source_federal_state_code,total_unit_count,mapped_unit_count,unresolved_unit_count,unit_mapping_rate,total_capacity_kw,mapped_capacity_kw,unresolved_capacity_kw,capacity_mapping_rate,cross_state_assignment_count
0,B,33180,33148,32,0.999036,338393.015,337985.929,407.086,0.998797,156
1,K,41490,41400,90,0.997831,498015.373,496642.953,1372.420,0.997244,17
2,NO,165317,164579,738,0.995536,1733380.203,1723683.882,9696.321,0.994406,842
3,OO,119381,118739,642,0.994622,1595002.200,1586315.625,8686.575,0.994554,193
4,S,33878,33763,115,0.996605,414791.167,413455.867,1335.300,0.996781,293
5,ST,94534,93309,1225,0.987042,1031355.419,1017848.051,13507.368,0.986903,147
6,T,38935,38891,44,0.998870,498239.215,497554.740,684.475,0.998626,21
7,V,23933,23893,40,0.998329,297799.696,297325.936,473.760,0.998409,17
8,W,18349,18331,18,0.999019,199881.246,199695.301,185.945,0.999070,143


,source_federal_state_code,derivation_method,unit_count,installed_capacity_kw
0,B,exact_locality_name_postcode,31831,324533.991
1,B,exact_municipality_name_postcode,707,7398.904
2,B,exact_national_locality_name_postcode,81,885.220
3,B,exact_national_municipality_name_postcode,75,567.280
4,B,reviewed_place_alias,85,757.865
...,...,...,...,...
63,V,vienna_postcode_and_source_or_name,2,14.240
64,W,exact_national_locality_name_postcode,133,2167.940
65,W,exact_national_municipality_name_postcode,10,95.595
66,W,unresolved,18,185.945


,source_federal_state_code,place_name_raw,postcode_raw,mapping_issue,unit_count,installed_capacity_kw
9,B,Neustift an der Lafnitz,7423.0,name_postcode_conflict,11,99.345
8,B,Neusiedl,7100.0,name_postcode_conflict,2,40.400
18,B,Wörterberg,8293.0,name_postcode_conflict,2,22.300
16,B,Steinbrunn Neue Siedlung,2491.0,NaN,2,14.760
4,B,Jennersdorf,8382.0,name_postcode_conflict,1,49.920
12,B,Oberloisdorf,7551.0,name_postcode_conflict,1,39.960
11,B,Nickelsdorf,2415.0,name_postcode_conflict,1,20.000
3,B,Draßburg/Rasporak,7021.0,NaN,1,17.670
14,B,Rohr im Burgenland,7551.0,name_postcode_conflict,1,17.500
2,B,Bocksdorf,7551.0,name_postcode_conflict,1,16.705


## 15. Aggregate photovoltaic data by municipality

In [14]:
resolved_units = mapped_units[
    mapped_units[
        "municipality_id"
    ].notna()
].copy()

pv_aggregated = (
    resolved_units
    .groupby(
        [
            "municipality_id",
            "reference_date",
        ],
        as_index=False,
    )
    .agg(
        unit_count=(
            "source_row_key",
            "size",
        ),
        installed_capacity_kw=(
            "installed_capacity_kw",
            lambda values: values.sum(
                min_count=1
            ),
        ),
        derivation_method=(
            "derivation_method",
            aggregate_mapping_method,
        ),
    )
)

municipality_grid = (
    municipality_master.copy()
)

municipality_grid[
    "reference_date"
] = REFERENCE_DATE

pv_harmonised = municipality_grid.merge(
    pv_aggregated,
    on=[
        "municipality_id",
        "reference_date",
    ],
    how="left",
)

pv_harmonised[
    "unit_count"
] = (
    pv_harmonised[
        "unit_count"
    ]
    .fillna(0)
    .astype(int)
)

zero_unit_mask = pv_harmonised[
    "unit_count"
].eq(0)

pv_harmonised.loc[
    zero_unit_mask,
    "installed_capacity_kw",
] = 0.0

pv_harmonised.loc[
    zero_unit_mask,
    "derivation_method",
] = "not_applicable"

pv_harmonised = pv_harmonised[
    [
        "municipality_id",
        "municipality_name",
        "federal_state_code",
        "reference_date",
        "unit_count",
        "derivation_method",
        "installed_capacity_kw",
    ]
].sort_values(
    "municipality_id"
).reset_index(drop=True)

pv_harmonised[
    "installed_capacity_kw"
] = pv_harmonised[
    "installed_capacity_kw"
].round(3)

assert len(pv_harmonised) == 2092

assert pv_harmonised[
    "municipality_id"
].is_unique

display(
    pv_harmonised.head()
)

,municipality_id,municipality_name,federal_state_code,reference_date,unit_count,derivation_method,installed_capacity_kw
0,10101,Eisenstadt,B,2026-07-18,1106,mixed,12250.020
1,10201,Rust,B,2026-07-18,203,mixed,1916.605
2,10301,Breitenbrunn am Neusiedler See,B,2026-07-18,309,mixed,2867.025
3,10302,Donnerskirchen,B,2026-07-18,243,mixed,2404.192
4,10303,Großhöflein,B,2026-07-18,228,mixed,2430.555


## 16. Validation

In [15]:
resolved_unit_count = len(
    resolved_units
)

aggregated_unit_count = int(
    pv_harmonised[
        "unit_count"
    ].sum()
)

resolved_capacity = resolved_units[
    "installed_capacity_kw"
].sum(min_count=1)

aggregated_capacity = pv_harmonised[
    "installed_capacity_kw"
].sum(min_count=1)

mapped_state_matches_gkz = (
    resolved_units[
        "federal_state_code"
    ]
    == resolved_units[
        "municipality_id"
    ].str[0].map(
        GKZ_PREFIX_TO_STATE_CODE
    )
).all()

validation_report = pd.DataFrame(
    [
        {
            "check": "municipality_count",
            "expected": 2092,
            "actual": len(
                pv_harmonised
            ),
            "passed": (
                len(pv_harmonised)
                == 2092
            ),
        },
        {
            "check": "aggregated_unit_count",
            "expected": (
                resolved_unit_count
            ),
            "actual": (
                aggregated_unit_count
            ),
            "passed": (
                resolved_unit_count
                == aggregated_unit_count
            ),
        },
        {
            "check": "aggregated_capacity_kw",
            "expected": (
                resolved_capacity
            ),
            "actual": (
                aggregated_capacity
            ),
            "passed": (
                pd.isna(
                    resolved_capacity
                )
                or np.isclose(
                    resolved_capacity,
                    aggregated_capacity,
                    rtol=0,
                    atol=0.01,
                )
            ),
        },
        {
            "check": "unique_municipality_ids",
            "expected": 2092,
            "actual": pv_harmonised[
                "municipality_id"
            ].nunique(),
            "passed": pv_harmonised[
                "municipality_id"
            ].is_unique,
        },
        {
            "check": "mapped_state_matches_gkz",
            "expected": True,
            "actual": (
                mapped_state_matches_gkz
            ),
            "passed": (
                mapped_state_matches_gkz
            ),
        },
        {
            "check": "approved_alias_keys_unique",
            "expected": True,
            "actual": (
                not approved_aliases.duplicated(
                    [
                        "source_federal_state_code",
                        "postcode",
                        "place_name_normalized",
                    ]
                ).any()
            ),
            "passed": (
                not approved_aliases.duplicated(
                    [
                        "source_federal_state_code",
                        "postcode",
                        "place_name_normalized",
                    ]
                ).any()
            ),
        },
    ]
)


# -------------------------------------------------------------------------
# Additional validation of the LV scope filter.
# -------------------------------------------------------------------------

scope_rows_expected = int(
    register_report[
        "pv_rows_before_lv_scope_filter"
    ].sum()
)

scope_rows_actual = int(
    register_report[
        "lv_proxy_rows"
    ].sum()
    + register_report[
        "excluded_above_100kw_rows"
    ].sum()
    + register_report[
        "invalid_capacity_rows"
    ].sum()
)

positive_capacity_expected = (
    register_report[
        "positive_capacity_kw_before_lv_scope_filter"
    ].sum()
)

positive_capacity_actual = (
    register_report[
        "lv_proxy_capacity_kw"
    ].sum()
    + register_report[
        "excluded_above_100kw_capacity_kw"
    ].sum()
)

included_capacity_max_100kw = bool(
    mapped_units[
        "installed_capacity_kw"
    ].le(
        LV_PROXY_MAX_BOTTLENECK_CAPACITY_KW
    ).all()
)

included_capacity_positive = bool(
    mapped_units[
        "installed_capacity_kw"
    ].gt(0).all()
)

scope_validation_rows = pd.DataFrame(
    [
        {
            "check": "lv_scope_row_partition",
            "expected": scope_rows_expected,
            "actual": scope_rows_actual,
            "passed": (
                scope_rows_expected
                == scope_rows_actual
            ),
        },
        {
            "check": "lv_scope_positive_capacity_partition",
            "expected": positive_capacity_expected,
            "actual": positive_capacity_actual,
            "passed": np.isclose(
                positive_capacity_expected,
                positive_capacity_actual,
                rtol=0,
                atol=0.01,
            ),
        },
        {
            "check": "included_capacity_max_100kw",
            "expected": True,
            "actual": included_capacity_max_100kw,
            "passed": included_capacity_max_100kw,
        },
        {
            "check": "included_capacity_positive",
            "expected": True,
            "actual": included_capacity_positive,
            "passed": included_capacity_positive,
        },
    ]
)

validation_report = pd.concat(
    [
        validation_report,
        scope_validation_rows,
    ],
    ignore_index=True,
)

assert validation_report[
    "passed"
].all(), (
    "At least one validation check failed."
)

display(
    validation_report
)

,check,expected,actual,passed
0,municipality_count,2092,2092,True
1,aggregated_unit_count,566053,566053,True
2,aggregated_capacity_kw,6570508.284,6570508.284,True
3,unique_municipality_ids,2092,2092,True
4,mapped_state_matches_gkz,True,True,True
5,approved_alias_keys_unique,True,True,True
6,lv_scope_row_partition,581075,581075,True
7,lv_scope_positive_capacity_partition,11611153.767,11611153.767,True
8,included_capacity_max_100kw,True,True,True
9,included_capacity_positive,True,True,True


## 17. Export harmonised outputs

In [16]:
plant_output_columns = [
    "source_file",
    "source_row_number",
    "source_register_id",
    "source_row_key",
    "postcode_raw",
    "place_name_raw",
    "capacity_raw",
    "postcode",
    "place_name",
    "place_name_normalized",
    "source_federal_state_code",
    "federal_state_code",
    "reference_date",
    "municipality_id",
    "municipality_name",
    "derivation_method",
    "mapping_issue",
    "cross_state_assignment",
    "installed_capacity_kw",
    "lv_scope_status",
    "lv_scope_reason",
]

plant_export = mapped_units[
    plant_output_columns
].copy()

text_columns = [
    "source_file",
    "source_register_id",
    "source_row_key",
    "postcode_raw",
    "place_name_raw",
    "capacity_raw",
    "postcode",
    "place_name",
    "place_name_normalized",
    "source_federal_state_code",
    "federal_state_code",
    "municipality_id",
    "municipality_name",
    "derivation_method",
    "mapping_issue",
    "lv_scope_status",
    "lv_scope_reason",
]

for column in text_columns:
    plant_export[column] = (
        plant_export[column]
        .astype("string")
    )

plant_export[
    "source_row_number"
] = pd.to_numeric(
    plant_export[
        "source_row_number"
    ],
    errors="coerce",
).astype("Int64")

plant_export[
    "installed_capacity_kw"
] = pd.to_numeric(
    plant_export[
        "installed_capacity_kw"
    ],
    errors="coerce",
).astype("float64")

plant_export[
    "cross_state_assignment"
] = plant_export[
    "cross_state_assignment"
].astype("boolean")

plant_export[
    "reference_date"
] = pd.to_datetime(
    plant_export[
        "reference_date"
    ],
    errors="coerce",
)

plant_export.to_parquet(
    PLANT_OUTPUT_PATH,
    index=False,
)

unresolved_export = plant_export[
    plant_export[
        "municipality_id"
    ].isna()
].copy()

audit_columns = [
    "source_file",
    "source_row_number",
    "source_register_id",
    "postcode_raw",
    "place_name_raw",
    "capacity_raw",
    "source_federal_state_code",
    "installed_capacity_kw",
    "lv_scope_status",
    "lv_scope_reason",
]

if not excluded_above_lv_capacity_rows.empty:
    excluded_above_100kw_export = (
        excluded_above_lv_capacity_rows[
            audit_columns
        ].copy()
    )
else:
    excluded_above_100kw_export = pd.DataFrame(
        columns=audit_columns
    )

if not invalid_capacity_rows.empty:
    invalid_capacity_export = (
        invalid_capacity_rows[
            audit_columns
        ].copy()
    )
else:
    invalid_capacity_export = pd.DataFrame(
        columns=audit_columns
    )

with pd.ExcelWriter(
    OUTPUT_XLSX_PATH,
    engine="openpyxl",
) as writer:
    pv_harmonised.to_excel(
        writer,
        sheet_name="pv_by_municipality",
        index=False,
    )

    lv_scope_summary.to_excel(
        writer,
        sheet_name="lv_scope_summary",
        index=False,
    )

    state_mapping_report.to_excel(
        writer,
        sheet_name="state_mapping_report",
        index=False,
    )

    mapping_summary.to_excel(
        writer,
        sheet_name="mapping_summary",
        index=False,
    )

    unresolved_top_places.to_excel(
        writer,
        sheet_name="unresolved_top_places",
        index=False,
    )

    unresolved_export.to_excel(
        writer,
        sheet_name="unresolved_units",
        index=False,
    )

    invalid_capacity_export.to_excel(
        writer,
        sheet_name="invalid_capacity",
        index=False,
    )

    excluded_above_100kw_export.to_excel(
        writer,
        sheet_name="excluded_above_100kw",
        index=False,
    )

    register_report.to_excel(
        writer,
        sheet_name="source_report",
        index=False,
    )

    technology_summary.to_excel(
        writer,
        sheet_name="technology_summary",
        index=False,
    )

    if not excluded_solar_like_rows.empty:
        excluded_solar_like_rows.to_excel(
            writer,
            sheet_name="excluded_solar_like",
            index=False,
        )

    approved_aliases.to_excel(
        writer,
        sheet_name="approved_review_mappings",
        index=False,
    )

    validation_report.to_excel(
        writer,
        sheet_name="validation",
        index=False,
    )

print(
    "Created:",
    OUTPUT_XLSX_PATH.resolve(),
)
print(
    "Created:",
    PLANT_OUTPUT_PATH.resolve(),
)
print(
    "Review workbook:",
    PLACE_REVIEW_PATH.resolve(),
)

Created: C:\Users\friha\TU Wien\Diplomarbeit\code\at-lv-scenario-benchmark\harmonisation\harmonisation_PV\outputs\pv_municipality_harmonised.xlsx
Created: C:\Users\friha\TU Wien\Diplomarbeit\code\at-lv-scenario-benchmark\harmonisation\harmonisation_PV\outputs\pv_plants_harmonised.parquet
Review workbook: C:\Users\friha\TU Wien\Diplomarbeit\code\at-lv-scenario-benchmark\harmonisation\harmonisation_PV\outputs\place_name_alias_review.xlsx


## 18. Final summary

In [17]:
total_capacity = mapped_units[
    "installed_capacity_kw"
].sum(min_count=1)

lv_summary = lv_scope_summary.iloc[0]

print(
    "PV rows before LV scope filter:",
    int(
        lv_summary[
            "pv_rows_before_lv_scope_filter"
        ]
    ),
)

print(
    "PV rows retained in LV proxy scope:",
    int(
        lv_summary[
            "lv_proxy_rows"
        ]
    ),
)

print(
    "PV rows excluded above 100 kW:",
    int(
        lv_summary[
            "excluded_above_100kw_rows"
        ]
    ),
)

print(
    "PV rows with invalid capacity:",
    int(
        lv_summary[
            "invalid_capacity_rows"
        ]
    ),
)

print(
    "LV proxy unit share:",
    f"{lv_summary['lv_proxy_unit_share']:.2%}",
)

print(
    "LV proxy capacity share:",
    f"{lv_summary['lv_proxy_capacity_share']:.2%}",
)

print(
    "Excluded capacity above 100 kW [MW]:",
    f"{lv_summary['excluded_above_100kw_capacity_kw'] / 1000:.3f}",
)

print(
    "Municipalities:",
    len(pv_harmonised),
)

print(
    "PV rows in LV proxy scope:",
    len(mapped_units),
)

print(
    "Mapped PV rows:",
    len(resolved_units),
)

print(
    "Unresolved PV rows:",
    len(unresolved_units),
)

print(
    "Unit mapping rate:",
    f"{len(resolved_units) / len(mapped_units):.2%}",
)

print(
    "Capacity mapping rate:",
    f"{resolved_capacity / total_capacity:.2%}",
)

print(
    "Approved reviewed mappings:",
    len(approved_aliases),
)

print(
    "Cross-state assignments:",
    mapped_units[
        "cross_state_assignment"
    ].sum(),
)

print(
    "Name/postcode conflicts:",
    mapped_units[
        "mapping_issue"
    ]
    .eq(
        "name_postcode_conflict"
    )
    .sum(),
)

PV rows before LV scope filter: 581075
PV rows retained in LV proxy scope: 568997
PV rows excluded above 100 kW: 11769
PV rows with invalid capacity: 309
LV proxy unit share: 97.92%
LV proxy capacity share: 56.90%
Excluded capacity above 100 kW [MW]: 5004.296
Municipalities: 2092
PV rows in LV proxy scope: 568997
Mapped PV rows: 566053
Unresolved PV rows: 2944
Unit mapping rate: 99.48%
Capacity mapping rate: 99.45%
Approved reviewed mappings: 937
Cross-state assignments: 1829
Name/postcode conflicts: 1871
